Task A Stage 1.5：续接验证

保留 Stage 1 / v2 文件及旧结果；直接读取原68条标注，不运行数据生成单元。执行19项原回归、8项v2评分回归，再验证v3授权调整与新增自由文本样例。

In [ ]:
from pathlib import Path
import json,hashlib
ROOT=Path.cwd()
if not (ROOT/'dual_register_spec.ipynb').exists():ROOT=ROOT/'Experiments'/'task_a'
for manifest_name,key in [('cpu_validation_manifest.json','files_sha256'),('free_text_audit_results.json','sha256')]:
    manifest=json.loads((ROOT/manifest_name).read_text())
    for name,digest in manifest[key].items():
        assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest, name

def load_definitions(name,tag):
    for cell in json.loads((ROOT/name).read_text())['cells']:
        if cell['cell_type']=='code' and tag in cell.get('metadata',{}).get('tags',[]):
            exec(compile(''.join(cell['source']),name,'exec'),globals())
load_definitions('dual_register_spec.ipynb','task_a_definitions')
old=json.loads((ROOT/'cpu_validation_tests.ipynb').read_text())
exec(compile(''.join(old['cells'][2]['source']),'stage1_regression','exec'),globals())
STAGE1_TESTS=TEST_RESULTS.copy()
# 直接读取原68个样例，不覆盖原样例或原结果。
ORIGINAL_CASES=[json.loads(line) for line in (ROOT/'parser_adversarial_cases.jsonl').read_text().splitlines()]
assert len(ORIGINAL_CASES)==68
CASES=ORIGINAL_CASES
load_definitions('free_text_scoring.ipynb','scoring_definitions')
old_tests=json.loads((ROOT/'free_text_scoring_tests.ipynb').read_text())
exec(compile(''.join(old_tests['cells'][3]['source']),'v2_regression','exec'),globals())
V2_TESTS=AUDIT_TESTS.copy()
load_definitions('free_text_scoring_v3.ipynb','scoring_definitions')


In [ ]:
import copy
# 原68条不重建；仅 duplicate_markers 的期望随用户授权修改，旧文件保留。
CASES=copy.deepcopy(ORIGINAL_CASES)
for c in CASES:
    if c['id']=='duplicate_markers':
        c['expected_answer']=[-12,3]
        c['expected_answer_ok']=True
        c['expectation_change']='用户批准：完整重复最终声明一致时去重；v2期望保留在原文件'

def add(name,raw,answer,ok,*,finish='stop',ambiguous=False):
    CASES.append({'id':name,'raw_response':raw,'gold':[-12,3],'mode':'DA',
                  'finish_reason':finish,'expected_answer':answer,'expected_answer_ok':ok,
                  'expected_da':{},'expected_ambiguity':ambiguous})

add('v3_duplicate_different_format','Final answer: (-12,3)\n最终答案：A=-12, B=3',[-12,3],True)
add('v3_duplicate_multiline','Final answer:\nA=-12\nB=3\nFinal answer:\n(-12,3)',[-12,3],True)
add('v3_duplicate_wrong_answer','Final answer: (3,-12)\nFinal answer: (3,-12)',[3,-12],False)
add('v3_three_conflicting_finals','Final answer: (-12,3)\nFinal answer: (3,-12)\nFinal answer: (-12,3)',None,False,ambiguous=True)
add('v3_incomplete_second_declaration','Final answer: (-12,3)\nFinal answer: (-12,',None,False,finish='length')
add('v3_marker_after_intermediate','Step 1: (-12,3)\nFinal answer:\n(3,-12)',[3,-12],False)
add('v3_empty_marker_after_intermediate','Step 1: (-12,3)\nFinal answer:\n',None,False)
add('v3_markdown_heading','### Final answer\n(-12,3)',[-12,3],True)
add('v3_markdown_heading_colon','### Final answer: **(-12,3)**',[-12,3],True)
add('v3_multiline_bold','**Final answer:**\n**A=-12**\n**B=3**',[-12,3],True)
add('v3_marker_body_inline_code','Final answer: `(-12,3)`',[-12,3],True)
add('v3_complete_at_limit_multiline','Final answer:\nA=-12\nB=3',[-12,3],True,finish='length')
add('v3_truncated_after_complete_prose','Final answer: (-12,3)\nI have completed the',[-12,3],True,finish='length')
add('v3_truncated_retraction','Final answer: (-12,3)\nWait, this is wrong',None,False,finish='length')
add('v3_truncated_new_candidate','Final answer: (-12,3)\nMaybe (3,',None,False,finish='length')
add('v3_truncated_unmarked_multiline','A=-12\nB=3',None,False,finish='length')
add('v3_normal_stop_incomplete','Final answer: (-12,',None,False)
add('v3_inline_alternatives','Final answer: (-12,3) or (3,-12)',None,False,ambiguous=True)
add('v3_unmarked_multiple','(-12,3)\n(3,-12)',None,False,ambiguous=True)
add('v3_other_numbers_in_trace','Step 1: A=41, B=18; u=7, v=3.\nStep 2: A=34, B=21.\nFinal answer: (-12,3)',[-12,3],True)
add('v3_zero_component','Final answer: (0,3)',[0,3],False)
add('v3_swapped_labels','Final answer: B=3, A=-12',None,False)

add('v3_inline_bold_labels','Final answer: **A**=-12, **B**=3',[-12,3],True)
add('v3_multiline_bold_labels','**Final answer**\n**A**=-12\n**B**=3',[-12,3],True)

class V3PolicyTests(unittest.TestCase):
    def test_blank_before_final_with_competing_pair(self):
        for prefix in ('\n', '\n\n', 'Step 1: (-12,3)\n\n'):
            with self.subTest(prefix=prefix):
                raw=prefix+'Final answer: (-12,3)\n(3,-12)'
                r=score_free_text(raw,[-12,3],'DA','stop')
                self.assertFalse(r['answer_parse_ok'])
                self.assertEqual(r['parse_reason'],'unlabelled_competing_answer')
                self.assertTrue(r['ambiguity'])
                self.assertFalse(r['answer_ok'])

    def test_blank_after_multiline_final_with_competing_pair(self):
        for prefix in ('', '\n', '\n\n'):
            with self.subTest(prefix=prefix):
                raw=prefix+'Final answer:\nA=-12\nB=3\n\n(3,-12)'
                r=score_free_text(raw,[-12,3],'DA','stop')
                self.assertFalse(r['answer_parse_ok'])
                self.assertEqual(r['parse_reason'],'unlabelled_competing_answer')
                self.assertTrue(r['ambiguity'])
                self.assertFalse(r['answer_ok'])

    def test_blank_after_final_with_retraction(self):
        for prefix in ('', '\n', '\n\n'):
            for finish in ('stop','length'):
                with self.subTest(prefix=prefix,finish=finish):
                    raw=prefix+'Final answer: (-12,3)\n\nWait, this is wrong'
                    r=score_free_text(raw,[-12,3],'DA',finish)
                    self.assertFalse(r['answer_parse_ok'])
                    self.assertEqual(r['parse_reason'],'text_after_final_answer')
                    self.assertFalse(r['ambiguity'])
                    self.assertFalse(r['answer_ok'])

    def test_blank_after_final_with_safe_truncated_tail(self):
        for prefix in ('', '\n', '\n\n'):
            for body in ('Final answer: (-12,3)', 'Final answer:\n(-12,3)',
                         'Final answer:\nA=-12\nB=3'):
                with self.subTest(prefix=prefix,body=body):
                    raw=prefix+body+'\n\nI have completed the'
                    r=score_free_text(raw,[-12,3],'DA','length')
                    self.assertTrue(r['answer_parse_ok'])
                    self.assertEqual(r['parse_reason'],'explicit_final')
                    self.assertFalse(r['ambiguity'])
                    self.assertTrue(r['answer_ok'])
                    self.assertTrue(r['truncated'])
                    self.assertEqual(r['raw_response'],raw)
                    start=len(prefix)
                    self.assertEqual(r['answer_line_indices'],
                                     list(range(start,start+len(body.splitlines()))))

    def test_new_case_ambiguity_labels(self):
        for c in CASES:
            if 'expected_ambiguity' in c:
                with self.subTest(case=c['id']):
                    score=score_free_text(c['raw_response'],c['gold'],c['mode'],c['finish_reason'])
                    self.assertEqual(score['ambiguity'],c['expected_ambiguity'])

    def test_duplicate_declarations_are_not_leakage(self):
        s=score_free_text('Final answer: (-12,3)\nFinal answer: (-12,3)',[-12,3],'DA','stop')
        self.assertTrue(s['answer_ok'])
        self.assertEqual(s['final_declaration_count'],2)
        self.assertFalse(s['ambiguity'])
        self.assertFalse(s['da_extra_text'])
        self.assertFalse(s['da_leakage'])

    def test_component_and_diagnostic_denominators(self):
        example=example_scores()
        r=summarize_scores(example)
        self.assertEqual(r['n_total'],5)
        self.assertEqual(r['n_parsed'],3)
        self.assertEqual(r['n_correct'],1)
        self.assertEqual(r['main_accuracy'],1/5)
        self.assertEqual(r['parse_success_rate'],3/5)
        self.assertEqual(r['conditional_answer_accuracy'],1/3)
        self.assertEqual(r['ambiguity_rate'],1/5)
        self.assertEqual(r['parse_failure_rate'],2/5)
        self.assertEqual(r['truncation_rate'],1/5)
        self.assertEqual(r['da_leakage_rate'],1/5)
        self.assertEqual(r['a_accuracy'],2/5)
        self.assertEqual(r['b_accuracy'],2/5)
        self.assertEqual(r['a_conditional_accuracy'],2/3)
        self.assertEqual(r['b_conditional_accuracy'],2/3)
        self.assertFalse(example[3]['a_ok']);self.assertFalse(example[3]['b_ok'])

    def test_leakage_does_not_change_content_score(self):
        s=score_free_text('1 + 2 = 3\nFinal answer: (-12,3)',[-12,3],'DA','stop')
        self.assertTrue(s['da_leakage']);self.assertTrue(s['answer_ok'])
        c=score_free_text(s['raw_response'],[-12,3],'CoT','stop')
        self.assertTrue(c['answer_ok']);self.assertIsNone(c['da_leakage'])
        self.assertIsNone(summarize_scores([c])['da_leakage_rate'])

    def test_truncation_alias_unknown_and_nontruncated_malformed(self):
        for finish in ('stop','length',None):
            s=score_free_text('Final answer: (-12,',[-12,3],'DA',finish)
            self.assertFalse(s['answer_ok'])
            self.assertEqual(s['truncated'],s['truncation'])
            self.assertEqual(s['truncation'],None if finish is None else finish=='length')
        s=score_free_text('Final answer: (-12,3)\nI checked the',[-12,3],'DA','length')
        self.assertTrue(s['answer_ok']);self.assertTrue(s['truncated'])
        self.assertTrue(s['da_extra_text'])

    def test_conflicting_final_claims_are_not_reasoning(self):
        for text in ('Final answer: (1,2)\nFinal answer: (2,1)',
                     'Final answer:\nA=1\nB=2\nFinal answer:\nA=2\nB=1'):
            r=score_free_text(text,[1,2],'DA','stop')
            self.assertTrue(r['ambiguity']);self.assertFalse(r['answer_ok'])
            self.assertFalse(r['da_arith']);self.assertFalse(r['da_trace'])
            self.assertFalse(r['da_leakage'])
            # 解析失败的额外文本指标仍为真；不能与推理泄漏混同。
            self.assertTrue(r['da_extra_text'])
        r=score_free_text('1+1=2\nFinal answer: (1,2)\nFinal answer: (2,1)',[1,2],'DA','stop')
        self.assertTrue(r['da_leakage'])

    def test_mixed_scoring_versions_rejected(self):
        r=score_free_text('(1,2)',[1,2],'DA','stop')
        r['scoring_version']='task-a-free-text-v2'
        with self.assertRaises(ValueError): summarize_scores([r])


def example_scores():
    return [score_free_text(raw,[1,2],'DA',finish) for raw,finish in (
        ('1 + 1 = 2\nFinal answer: (1,2)','stop'),
        ('(1,3)','stop'), ('(9,2)','stop'),
        ('Final answer: (1,2)\nFinal answer: (2,1)','stop'), ('','length'))]

# 使用现有测试类及新策略测试，保留500组随机检查的原始种子与定义。
suite=unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(FreeTextAuditTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(V3PolicyTests)])
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'v3测试失败，停止输出成功报告'
V3_TESTS={'tests_run':result.testsRun,'failures':len(result.failures),'errors':len(result.errors),
          'skipped':len(result.skipped),'original_cases':len(ORIGINAL_CASES),
          'added_cases':len(CASES)-len(ORIGINAL_CASES),'total_cases':len(CASES),
          'randomized_groups':500,'randomized_response_checks':1000}


In [ ]:
import platform
results=[{**c,'actual':score_free_text(c['raw_response'],c['gold'],c['mode'],c['finish_reason'])} for c in CASES]
# 本轮只回读核对已有92条逐例结果，不改写历史 JSONL。
expected_results=''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in results)
assert (ROOT/'parser_case_results_v3.jsonl').read_text()==expected_results, '已有逐例结果发生变化'
# 明确记录唯一已授权的原样例期望变化；不覆盖原始68条标注。
correct=sum(r['actual']['parsed_answer']==r['expected_answer'] and r['actual']['answer_parse_ok']==(r['expected_answer'] is not None) and r['actual']['answer_ok']==r['expected_answer_ok'] for r in results)
report={'scoring_version':SCORING_VERSION,'purpose':'synthetic_parser_audit_only',
        'python':platform.python_version(),'stage1_regression':STAGE1_TESTS,'v2_regression':V2_TESTS,
        'v3_tests':V3_TESTS,'annotated_decisions_correct':correct,'annotated_decisions_total':len(results),
        'expectation_changes':[{'id':'duplicate_markers','v2':'parse_failure','v3':'accept_identical_final_pair','authorization':'用户本轮明确要求'}],
        'fixture_summary':summarize_scores([r['actual'] for r in results]),
        'calculation_example':{'rows':example_scores(),'summary':summarize_scores(example_scores())},
        'model_inference':'not_run','stage1_and_v2_hashes_verified':True,
        'sha256':{name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in
                  ('free_text_scoring_v3.ipynb','free_text_scoring_v3_tests.ipynb','parser_case_results_v3.jsonl')}}
assert correct==len(results)
(ROOT/'free_text_audit_results_v3.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:report[k] for k in ('stage1_regression','v2_regression','v3_tests','annotated_decisions_correct','annotated_decisions_total')},ensure_ascii=False,indent=2))
print(json.dumps(report['calculation_example']['summary'],ensure_ascii=False,indent=2))
